In [ ]:
import numpy as np

K = 2
T = 10_000
M = 20

seed = 2026
rng = np.random.default_rng(seed=seed)
pairs = [
    [.1,0]
    , [.2,.1]
    ,[.5,.4]
    ,[.4,.5]
    , [.7,.8]
    , [.9,1]
]

def sample_loss(rng, q):
    success = rng.binomial(n=1, p=q, size=1).item()
    return 1 if success else 0

In [ ]:
def eps_greedy(q1, q2, rng):
    smallest = np.argmin([q1,q2])
    delta = np.max([q1,q2])-np.min([q1,q2]) #expected loss is just q
    eps = 8*np.log(T*delta**2/4)/(T*delta**2)
    n = np.zeros(K)
    cumul_losses = np.zeros(K)
    epsT = round(eps*T)
    regret = 0
    sampled_losses=[] #useless but to remove annoying unbound warning

    for t in range(epsT+1):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        I_t = np.argmin(n)
        n[I_t] += 1
        cumul_losses[I_t] += sampled_losses[I_t]
        regret += sampled_losses[I_t] - sampled_losses[smallest]
    best = np.argmin(cumul_losses)
    for t in range(epsT+1, T):
        regret += sampled_losses[best] - sampled_losses[smallest]

    return regret


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    for i in range(M):
        regrets[i] = eps_greedy(
            q1=pair[0]
            , q2=pair[1]
            , rng=rng
        )
    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f}",flush=True)

print(f"{summed_avg_regret:>{14}.3f} {np.sqrt(var_summed_regret):>{10}.3f}",flush=True)

In [ ]:
def lcb1(q1, q2, delta, rng):
    smallest = np.argmin([q1,q2])

    #first we init the variables (t=1 to t=K)
    cumul_losses = np.zeros(K)
    n = np.repeat(1,K)
    regret = 0
    for t in range(K):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        regret += sampled_losses[t] - sampled_losses[smallest]
        cumul_losses[t] += sampled_losses[t]
    mu_hats = np.copy(cumul_losses)
    mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta))

    for t in range(K, T):
        I_t = np.argmin(mu_deltas)

        #update variables for next iter:
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[I_t] += sampled_losses[I_t]

        n[I_t] += 1
        mu_hats = cumul_losses/n
        mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta)/n)

        regret += sampled_losses[I_t] - sampled_losses[smallest]

    return regret


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    for i in range(M):
        regrets[i] = lcb1(
            q1=pair[0]
            , q2=pair[1]
            , delta=1/T**2
            , rng=rng
        )
    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f}",flush=True)

print(f"{summed_avg_regret:>{14}.3f} {np.sqrt(var_summed_regret):>{10}.3f}",flush=True)

In [ ]:
def lcb2(q1, q2, delta, rng):
    smallest = np.argmin([q1,q2])

    #first we init the variables (t=1 to t=K)
    n = np.repeat(1,K)
    regret = 0
    #the two lists below allow efficient computation of var hats
    cumul_losses = np.zeros(K)
    cumul_sq_losses = np.zeros(K)

    for t in range(K):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[t] += sampled_losses[t]
        cumul_sq_losses[t] += sampled_losses[t]**2
        regret += sampled_losses[t] - sampled_losses[smallest]
    mu_hats = np.copy(cumul_losses)
    var_hats = cumul_sq_losses/n - mu_hats**2 #var=E(x^2)-E(x)^2
    mu_deltas = mu_hats-np.sqrt(2*var_hats*np.log(1/delta)/n)-3*np.log(1/delta)/n

    for t in range(K, T):
        I_t = np.argmin(mu_deltas)

        #update variables for next iter:
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        cumul_losses[I_t] += sampled_losses[I_t]
        cumul_sq_losses[I_t] += sampled_losses[I_t]**2

        n[I_t] += 1
        mu_hats = cumul_losses/n
        var_hats = cumul_sq_losses/n - mu_hats**2
        mu_deltas = mu_hats-np.sqrt(2*var_hats*np.log(1/delta)/n)-3*np.log(1/delta)/n

        regret += sampled_losses[I_t] - sampled_losses[smallest]

    return regret


summed_avg_regret = 0
var_summed_regret = 0

print(f"{'pair':<{17}} {'avg regret':>{10}} {'sd':>{10}}",flush=True)
for pair in pairs:
    regrets = np.zeros(M)
    for i in range(M):
        regrets[i] = lcb2(
            q1=pair[0]
            , q2=pair[1]
            , delta=1/T**2
            , rng=rng
        )
    avg_regret = np.average(regrets)
    summed_avg_regret += avg_regret
    var_regret = np.var(regrets,ddof=1)
    var_summed_regret += var_regret #we assume all trials independent so we can just sum the variances
    print(f"{str(pair):<{17}} {avg_regret:>{10}.3f} {np.sqrt(var_regret):>{10}.3f}",flush=True)

print(f"{summed_avg_regret:>{14}.3f} {np.sqrt(var_summed_regret):>{10}.3f}",flush=True)